# LLM-as-judge for EMPI entity matching — speed comparison

Goal: run a small LLM as judge for whether two patient records are the same person, on **10 synthetic pairs**, comparing three execution strategies. Scaling target: ~200k blocked candidate pairs on a Windows VM (32 GB RAM, 8-core CPU, no GPU). Local dev is Apple M4 / Metal.

**Strategies measured (cells 6, 8, 9):**
1. **Single process × 8 threads** (baseline) — `Llama.create_chat_completion`.
2. **2 worker processes × 4 threads each** — `concurrent.futures.ProcessPoolExecutor`. Can beat single-process on memory-bandwidth-bound CPUs.
3. **`llama_cpp.server` + concurrent async client** — the deployment shape. Main win over baseline is reduced per-call Python overhead.

**Speed levers applied to every strategy:**
- Smaller model: `Llama-3.2-3B-Instruct` Q4_K_M (~2 GB) instead of Jellyfish-8B.
- KV-cache reuse for the shared system+instruction prefix.
- GBNF grammar pinning output to `Yes | No` (one sampled token, no off-task generation).
- EM-specific prompt with explicit guidance on strong identifiers, missing values, and noise tolerance.

In [11]:
!pip install -q "llama-cpp-python>=0.2.90" "huggingface_hub>=0.23" "httpx>=0.27" \
    "uvicorn>=0.22" "fastapi>=0.100" "sse-starlette>=1.6" "pydantic-settings>=2.0" \
    "starlette-context>=0.3.6"

In [ ]:
import os
from huggingface_hub import hf_hub_download

# ===== MODEL CONFIG =====
# Active: smaller, general-purpose instruct model. Faster, fits in ~2 GB on disk.

MODEL_REPO = "bartowski/Llama-3.2-3B-Instruct-GGUF"
MODEL_FILE = "Llama-3.2-3B-Instruct-Q4_K_M.gguf"

# Previous: Jellyfish-8B, instruction-tuned on EM tasks. Higher accuracy ceiling
# but ~2-3x slower per call. Kept for A/B comparison.
# MODEL_REPO = "mradermacher/Jellyfish-8B-i1-GGUF"
# MODEL_FILE = "Jellyfish-8B.i1-Q4_K_M.gguf"

# ===== RUNTIME CONFIG =====
N_CTX        = 1024            # the EM prompt fits well under this; less KV-cache RAM
N_THREADS    = os.cpu_count()  # full CPU per llama instance
N_GPU_LAYERS = -1              # Metal on Mac; no-op on CPU-only Windows wheel

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print(f"Model: {MODEL_REPO}/{MODEL_FILE}")
print(f"Cached at: {model_path}")

/opt/anaconda3/envs/llm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: bartowski/Llama-3.2-3B-Instruct-GGUF/Llama-3.2-3B-Instruct-Q4_K_M.gguf
Cached at: /Users/miguelroca/.cache/huggingface/hub/models--bartowski--Llama-3.2-3B-Instruct-GGUF/snapshots/5ab33fa94d1d04e903623ae72c95d1696f09f9e8/Llama-3.2-3B-Instruct-Q4_K_M.gguf


In [5]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    n_threads=N_THREADS,
    verbose=False,
)

llama_context: n_ctx_seq (1024) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


In [6]:
# 10 synthetic EMPI record pairs, mix of matches and non-matches.
# PATID is intentionally omitted from anything sent to the model (would leak the answer).
# `expected` is ground truth, used only for the accuracy report after inference.

EMPI_FIELDS = [
    "FirstNM", "LastNM", "MiddleNM", "SuffixNM", "BirthDT", "SSN",
    "AddressLine1", "AddressLine2", "CityNM", "StateCD", "ZipCD", "CountryNM",
    "PrimaryPhoneNBR", "Phone01NBR", "Phone02NBR", "Phone03NBR",
    "Email", "SexAtBirthDSC",
]

def rec(**kwargs):
    return {f: kwargs.get(f) for f in EMPI_FIELDS}

cases = [
    # 1. Match: middle name initial vs spelled out, address formatting, phone rotation, email case.
    dict(expected="Yes",
         a=rec(FirstNM="Maria", LastNM="Gonzalez", MiddleNM="Elena", BirthDT="1987-04-12",
               SSN="123-45-6789", AddressLine1="245 W North Avenue", AddressLine2="Apt 4B",
               CityNM="Chicago", StateCD="IL", ZipCD="60610", CountryNM="United States",
               PrimaryPhoneNBR="312-555-0142", Phone01NBR="773-555-0199",
               Email="maria.gonzalez@example.com", SexAtBirthDSC="Female"),
         b=rec(FirstNM="Maria", LastNM="Gonzalez", MiddleNM="E.", BirthDT="1987-04-12",
               SSN="123-45-6789", AddressLine1="245 West North Ave", AddressLine2="#4B",
               CityNM="Chicago", StateCD="IL", ZipCD="60610", CountryNM="USA",
               PrimaryPhoneNBR="773-555-0199", Phone01NBR="312-555-0142",
               Email="Maria.Gonzalez@example.com", SexAtBirthDSC="Female")),
    # 2. Match: nickname vs full first name.
    dict(expected="Yes",
         a=rec(FirstNM="Bob", LastNM="Smith-Jones", BirthDT="1975-11-03", SSN="222-33-4444",
               AddressLine1="100 Main St", CityNM="Evanston", StateCD="IL", ZipCD="60201",
               PrimaryPhoneNBR="847-555-0100", Email="bob.sj@example.org", SexAtBirthDSC="Male"),
         b=rec(FirstNM="Robert", LastNM="Smith-Jones", BirthDT="1975-11-03", SSN="222-33-4444",
               AddressLine1="100 Main Street", CityNM="Evanston", StateCD="IL", ZipCD="60201-1234",
               PrimaryPhoneNBR="(847) 555-0100", Email="robert.sj@example.org",
               SexAtBirthDSC="Male")),
    # 3. Match: maiden vs married name, same DOB and SSN.
    dict(expected="Yes",
         a=rec(FirstNM="Jennifer", LastNM="Park", BirthDT="1992-08-21", SSN="555-66-7777",
               AddressLine1="42 Oak Dr", CityNM="Naperville", StateCD="IL", ZipCD="60540",
               PrimaryPhoneNBR="630-555-0123", Email="jpark@example.com", SexAtBirthDSC="Female"),
         b=rec(FirstNM="Jennifer", LastNM="Park-Liu", BirthDT="1992-08-21", SSN="555-66-7777",
               AddressLine1="42 Oak Drive", CityNM="Naperville", StateCD="IL", ZipCD="60540",
               PrimaryPhoneNBR="630-555-0123", Email="jennifer.liu@example.com",
               SexAtBirthDSC="Female")),
    # 4. Match: typo in last name, missing email in one record.
    dict(expected="Yes",
         a=rec(FirstNM="Carlos", LastNM="Hernandez", BirthDT="1968-02-14", SSN="888-99-1111",
               AddressLine1="789 Pine St Apt 12", CityNM="Cicero", StateCD="IL", ZipCD="60804",
               PrimaryPhoneNBR="708-555-0188", Email="c.hernandez@example.com",
               SexAtBirthDSC="Male"),
         b=rec(FirstNM="Carlos", LastNM="Hernadez", BirthDT="1968-02-14", SSN="888-99-1111",
               AddressLine1="789 Pine Street, Apt 12", CityNM="Cicero", StateCD="IL", ZipCD="60804",
               PrimaryPhoneNBR="708-555-0188", SexAtBirthDSC="Male")),
    # 5. Match: same person moved, phones updated, SSN+DOB anchor.
    dict(expected="Yes",
         a=rec(FirstNM="Aisha", LastNM="Patel", BirthDT="2000-06-30", SSN="333-44-5555",
               AddressLine1="55 Elm Ct", CityNM="Skokie", StateCD="IL", ZipCD="60076",
               PrimaryPhoneNBR="847-555-0211", Email="aisha.p@example.net",
               SexAtBirthDSC="Female"),
         b=rec(FirstNM="Aisha", LastNM="Patel", BirthDT="2000-06-30", SSN="333-44-5555",
               AddressLine1="1820 N Clark", AddressLine2="Apt 9", CityNM="Chicago", StateCD="IL",
               ZipCD="60614", PrimaryPhoneNBR="312-555-0333", Email="aisha.p@example.net",
               SexAtBirthDSC="Female")),
    # 6. Non-match: similar names, different DOB and SSN.
    dict(expected="No",
         a=rec(FirstNM="John", LastNM="Smith", BirthDT="1980-01-15", SSN="111-22-3333",
               AddressLine1="1 First Ave", CityNM="Chicago", StateCD="IL", ZipCD="60601",
               PrimaryPhoneNBR="312-555-0001", Email="john.smith@example.com",
               SexAtBirthDSC="Male"),
         b=rec(FirstNM="John", LastNM="Smith", BirthDT="1955-07-22", SSN="999-88-7777",
               AddressLine1="500 Second St", CityNM="Chicago", StateCD="IL", ZipCD="60602",
               PrimaryPhoneNBR="312-555-9999", Email="jsmith2@example.com",
               SexAtBirthDSC="Male")),
    # 7. Non-match: father/son same name, similar address.
    dict(expected="No",
         a=rec(FirstNM="Michael", LastNM="O'Brien", SuffixNM="Sr", BirthDT="1955-03-10",
               SSN="444-55-6666", AddressLine1="22 Lake Shore Dr", CityNM="Chicago",
               StateCD="IL", ZipCD="60611", PrimaryPhoneNBR="312-555-0444",
               Email="m.obrien@example.com", SexAtBirthDSC="Male"),
         b=rec(FirstNM="Michael", LastNM="O'Brien", SuffixNM="Jr", BirthDT="1985-09-22",
               SSN="666-77-8888", AddressLine1="22 Lake Shore Dr", CityNM="Chicago",
               StateCD="IL", ZipCD="60611", PrimaryPhoneNBR="312-555-0445",
               Email="mike.obrien@example.com", SexAtBirthDSC="Male")),
    # 8. Non-match: twins — same DOB and last name, different first name and SSN.
    dict(expected="No",
         a=rec(FirstNM="Emma", LastNM="Wright", BirthDT="2010-12-05", SSN="121-21-2121",
               AddressLine1="9 Birch Ln", CityNM="Oak Park", StateCD="IL", ZipCD="60302",
               PrimaryPhoneNBR="708-555-0011", Email="parent@example.com",
               SexAtBirthDSC="Female"),
         b=rec(FirstNM="Olivia", LastNM="Wright", BirthDT="2010-12-05", SSN="121-21-2122",
               AddressLine1="9 Birch Ln", CityNM="Oak Park", StateCD="IL", ZipCD="60302",
               PrimaryPhoneNBR="708-555-0011", Email="parent@example.com",
               SexAtBirthDSC="Female")),
    # 9. Non-match: completely different people.
    dict(expected="No",
         a=rec(FirstNM="Linda", LastNM="Nguyen", BirthDT="1990-05-17", SSN="777-66-5555",
               AddressLine1="3300 Sheridan Rd", CityNM="Chicago", StateCD="IL", ZipCD="60657",
               PrimaryPhoneNBR="773-555-0777", Email="lnguyen@example.com",
               SexAtBirthDSC="Female"),
         b=rec(FirstNM="Tomasz", LastNM="Kowalski", BirthDT="1962-11-08", SSN="012-34-5678",
               AddressLine1="8 Pulaski Rd", CityNM="Chicago", StateCD="IL", ZipCD="60624",
               PrimaryPhoneNBR="773-555-0808", Email="tk@example.com", SexAtBirthDSC="Male")),
    # 10. Non-match: SSN typo collision, conflicting names and DOB.
    dict(expected="No",
         a=rec(FirstNM="Sarah", LastNM="Cohen", BirthDT="1978-10-02", SSN="234-56-7890",
               AddressLine1="14 Howard St", CityNM="Evanston", StateCD="IL", ZipCD="60202",
               PrimaryPhoneNBR="847-555-0234", Email="sarah.cohen@example.com",
               SexAtBirthDSC="Female"),
         b=rec(FirstNM="David", LastNM="Brown", BirthDT="1945-04-19", SSN="234-56-7890",
               AddressLine1="900 Roosevelt Rd", CityNM="Berwyn", StateCD="IL", ZipCD="60402",
               PrimaryPhoneNBR="708-555-0345", Email="d.brown@example.com",
               SexAtBirthDSC="Male")),
]

n_yes = sum(c["expected"] == "Yes" for c in cases)
print(f"{len(cases)} cases ready ({n_yes} match / {len(cases) - n_yes} non-match).")

10 cases ready (5 match / 5 non-match).


In [7]:
# === Refined EM prompt + single-process timed run (baseline #1) ===

import time
from llama_cpp import LlamaRAMCache
from llama_cpp.llama_grammar import LlamaGrammar

SYSTEM_MSG = (
    "You are an expert at matching patient records. "
    "Decide whether two records describe the same person. "
    "Answer with exactly one word: Yes or No."
)

INSTRUCTION_PREFIX = (
    "Compare the two patient records and decide if they refer to the same person.\n"
    "\n"
    "Guidelines:\n"
    "- Strong identifiers: SSN, BirthDT, full name. Matching SSN and BirthDT with "
    "similar names almost always means the same person.\n"
    "- Tolerate noise: typos, nicknames (Bob/Robert), case, punctuation, "
    "address abbreviations (St/Street), reordered phone numbers, formatting "
    "differences (USA/United States).\n"
    "- A missing value (N/A) is NOT evidence against a match.\n"
    "- Conflicting non-null SSNs or BirthDTs indicate different people.\n"
    "- Suffix mismatch (Sr vs Jr) usually indicates parent vs child, not the same person.\n"
    "\n"
)

def serialize(rec):
    return ", ".join(
        f"{k}: {('N/A' if v in (None, '', 'nan') else v)}" for k, v in rec.items()
    )

def build_user_msg(a, b):
    return (
        INSTRUCTION_PREFIX
        + f"Record A: [{serialize(a)}]\n"
        + f"Record B: [{serialize(b)}]\n"
        + "Same person? Answer Yes or No."
    )

llm.cache = LlamaRAMCache(capacity_bytes=512 * 1024 * 1024)
yesno_grammar = LlamaGrammar.from_string('root ::= "Yes" | "No"', verbose=False)

def classify(a, b):
    out = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": build_user_msg(a, b)},
        ],
        max_tokens=2,
        temperature=0.0,
        grammar=yesno_grammar,
    )
    return out["choices"][0]["message"]["content"].strip()

print("Warm-up...")
t0 = time.perf_counter()
_ = classify(cases[0]["a"], cases[0]["b"])
print(f"  warm-up: {time.perf_counter() - t0:.2f}s\n")

print(f"{'#':>2}  {'expected':>8}  {'predicted':>9}  {'time (s)':>9}  result")
print("-" * 50)
per_case_single = []
correct_single = 0
t_total = time.perf_counter()
for i, c in enumerate(cases, 1):
    t0 = time.perf_counter()
    pred = classify(c["a"], c["b"])
    dt = time.perf_counter() - t0
    per_case_single.append(dt)
    ok = pred == c["expected"]
    correct_single += ok
    print(f"{i:>2}  {c['expected']:>8}  {pred:>9}  {dt:>9.3f}  {'OK' if ok else 'WRONG'}")
total_single = time.perf_counter() - t_total
print("-" * 50)
print(f"[SINGLE] total {total_single:.2f}s | mean {total_single/len(cases):.3f}s/pair | "
      f"accuracy {correct_single}/{len(cases)}")

Warm-up...
  warm-up: 1.22s

 #  expected  predicted   time (s)  result
--------------------------------------------------
 1       Yes        Yes      0.101  OK
 2       Yes        Yes      0.760  OK
 3       Yes         No      0.792  WRONG
 4       Yes        Yes      0.776  OK
 5       Yes        Yes      0.772  OK
 6        No         No      0.814  OK
 7        No         No      0.787  OK
 8        No         No      0.726  OK
 9        No         No      0.782  OK
10        No         No      0.792  OK
--------------------------------------------------
[SINGLE] total 7.10s | mean 0.710s/pair | accuracy 9/10
